In [1]:
import pandas as pd
import glob

files = glob.glob("data/raw/*.csv")
print(files)

# Load one file first to see the actual structure
sample = pd.read_csv("data/raw/raw_test_bernie.csv")
print(sample.shape)
print(sample.columns.tolist())
print(sample.head())

['data/raw/raw_test_trump.csv', 'data/raw/raw_test_biden.csv', 'data/raw/raw_train_biden.csv', 'data/raw/raw_val_trump.csv', 'data/raw/raw_train_bernie.csv', 'data/raw/raw_val_bernie.csv', 'data/raw/raw_val_biden.csv', 'data/raw/raw_test_bernie.csv', 'data/raw/raw_train_trump.csv']
(635, 3)
['Tweet', 'Target', 'Stance']
                                               Tweet          Target   Stance
0  #IEndorseBernie for tons of reasons, but this ...  Bernie Sanders    FAVOR
1  A big problem w/#Bernie left is not only preoc...  Bernie Sanders  AGAINST
2  This poll is not reflecting anything: "age was...  Bernie Sanders  AGAINST
3  So proud how @BernieSanders is shedding light ...  Bernie Sanders    FAVOR
4  According to media bias fact checker, you have...  Bernie Sanders    FAVOR


In [7]:
splits = ['train', 'val', 'test']
targets = ['trump', 'biden', 'bernie']

count = 0

for split in splits:
    for target in targets:
        df = pd.read_csv(f"data/raw/raw_{split}_{target}.csv")
        print(f"{split}_{target}: {df.shape[0]} rows")
        print(df['Stance'].value_counts()) 
        print()

        count += df.shape[0]

print(f"count {count}")

train_trump: 6362 rows
Stance
AGAINST    3425
FAVOR      2937
Name: count, dtype: int64

train_biden: 5806 rows
Stance
AGAINST    3254
FAVOR      2552
Name: count, dtype: int64

train_bernie: 5056 rows
Stance
FAVOR      2858
AGAINST    2198
Name: count, dtype: int64

val_trump: 814 rows
Stance
AGAINST    440
FAVOR      374
Name: count, dtype: int64

val_biden: 745 rows
Stance
AGAINST    417
FAVOR      328
Name: count, dtype: int64

val_bernie: 634 rows
Stance
FAVOR      350
AGAINST    284
Name: count, dtype: int64

test_trump: 777 rows
Stance
AGAINST    425
FAVOR      352
Name: count, dtype: int64

test_biden: 745 rows
Stance
AGAINST    408
FAVOR      337
Name: count, dtype: int64

test_bernie: 635 rows
Stance
FAVOR      343
AGAINST    292
Name: count, dtype: int64

count 21574


In [8]:
train_dfs = []
for target in targets:
    df = pd.read_csv(f"data/raw/raw_train_{target}.csv")
    train_dfs.append(df)

train_combined = pd.concat(train_dfs, ignore_index=True)
train_combined.to_csv("data/processed/train_combined.csv", index=False)

In [9]:
for split in ['train', 'val', 'test']:
    for target in ['trump', 'biden', 'bernie']:
        df = pd.read_csv(f"data/raw/raw_{split}_{target}.csv")
        print(f"{split}_{target}:")
        print(df['Stance'].value_counts(normalize=True))
        print()

train_trump:
Stance
AGAINST    0.538353
FAVOR      0.461647
Name: proportion, dtype: float64

train_biden:
Stance
AGAINST    0.560455
FAVOR      0.439545
Name: proportion, dtype: float64

train_bernie:
Stance
FAVOR      0.565269
AGAINST    0.434731
Name: proportion, dtype: float64

val_trump:
Stance
AGAINST    0.540541
FAVOR      0.459459
Name: proportion, dtype: float64

val_biden:
Stance
AGAINST    0.559732
FAVOR      0.440268
Name: proportion, dtype: float64

val_bernie:
Stance
FAVOR      0.55205
AGAINST    0.44795
Name: proportion, dtype: float64

test_trump:
Stance
AGAINST    0.546976
FAVOR      0.453024
Name: proportion, dtype: float64

test_biden:
Stance
AGAINST    0.547651
FAVOR      0.452349
Name: proportion, dtype: float64

test_bernie:
Stance
FAVOR      0.540157
AGAINST    0.459843
Name: proportion, dtype: float64



In [10]:
train = pd.read_csv("data/processed/train_combined.csv")
test_dfs = [pd.read_csv(f"data/raw/raw_test_{t}.csv") for t in ['trump','biden','bernie']]
test = pd.concat(test_dfs, ignore_index=True)

overlap = set(train['Tweet']) & set(test['Tweet'])
print(len(overlap))

1


In [12]:
from transformers import AutoTokenizer
from src.data import format_prompt

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

lengths = train.apply(lambda row: len(tokenizer.encode(format_prompt(row['Tweet'], row['Target']))), axis=1)
print(lengths.describe())

count    17224.000000
mean        67.619194
std         16.784980
min         33.000000
25%         54.000000
50%         67.000000
75%         81.000000
max        158.000000
dtype: float64


In [1]:
!nvidia-smi

Wed Aug 12 03:43:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!git clone https://github.com/meghna-adduri/political-stance-detection.git

Cloning into 'political-stance-detection'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 57 (delta 14), reused 51 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 2.95 MiB | 18.74 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [9]:
%cd political-stance-detection

/content/political-stance-detection


In [10]:
!pwd
!ls

/content/political-stance-detection
data	   pyproject.toml  requirements.txt  src
notebooks  README.md	   scratch.ipynb     tests


In [6]:
%cd /content/political-stance-detection

[Errno 2] No such file or directory: '/content/political-stance-detection'
/content


In [11]:
!pip install transformers accelerate bitsandbytes wandb -q

In [12]:
import wandb
wandb.login()

True

In [ ]:
from src.baseline import load_model, run_baseline, evaluate_and_log

model, tokenizer = load_model()
results = run_baseline(model, tokenizer)
acc, macro_f1 = evaluate_and_log(results)
print(f"Zero-shot accuracy: {acc:.3f}, macro-F1: {macro_f1:.3f}")

ModuleNotFoundError: No module named 'src'